In [1]:
# Import knižníc potrebných na model, spracovanie obrázkov a vyhodnotenie experimentu.
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import random
import math
import numpy as np
import copy
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score, precision_score, recall_score
import cv2
from PIL import Image
import glob
import os

In [2]:
# Základné parametre experimentu: veľkosť vstupu, batch size, počet epoch a learning rate.
IMG_HEIGHT = 256 
IMG_WIDTH = 256
batch_size = 16
epochs = 100
learning_rate = 1e-3
patience = 12

In [3]:
# Definícia autoencodera. Model sa učí rekonštruovať dobré vzorky.
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.encoder = nn.Sequential(
            # (0)-(1)
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (2)-(3)
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (4)-(5)
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (6)-(7)
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (8)-(9)
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (10)
            nn.Conv2d(512, 16, kernel_size=3, stride=1, padding=1)
        )
        
        self.decoder = nn.Sequential(
            # (0)-(1)
            nn.ConvTranspose2d(16, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            # (2)-(3)
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (4)-(5)
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (6)-(7)
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (8)-(9)
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            # (10)-(11)
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Autoencoder().to(device)

In [ ]:
# Vytvorenie predspracovaných datasetov. Porovnávajú sa dve metódy: HSV úprava a inpainting.
import os
import cv2
import numpy as np
from PIL import Image
import glob
import random
import matplotlib.pyplot as plt

# 1. DEFINÍCIA FUNKCIÍ
#    - Inpaint: maska len pre veľmi silné odlesky > 240
#    - HSV: zrazenie jasu nad 220 na hodnotu 220
def remove_glare_inpaint(img_rgb):
    """
    Detekcia iba extrémnych odleskov (> 240) pre metódu Inpaint.
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    # Maska iba pre veľmi svetlé odlesky
    _, mask_glare = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)

    # Odstránenie malého šumu z masky
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask_glare = cv2.morphologyEx(mask_glare, cv2.MORPH_OPEN, kernel_open)

    # Jemné rozšírenie masky okolo odlesku
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask_dilated = cv2.dilate(mask_glare, kernel_dilate)

    if np.count_nonzero(mask_dilated) == 0:
        return img_rgb.copy(), mask_dilated

    # Inpainting opraví len vypálené svetlé miesta
    result = cv2.inpaint(img_rgb, mask_dilated, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    return result, mask_dilated


def remove_glare_hsv(img_rgb):
    """
    Zráža extrémne svetlé miesta:
    všetko nad 220 nastaví na 220.
    """
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    V = hsv[:, :, 2]

    # Limitácia horného jasu
    V[V > 220] = 220

    hsv[:, :, 2] = V
    result = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return result


# 2. HLAVNÝ SKRIPT NA VYTVORENIE DATASETU
def generate_datasets(base_source_dir, target_root_hsv, target_root_inpaint):
    print("--- ŠTART GENEROVANIA DATASETOV (INPAINT > 240, HSV > 220) ---")

    for split in ['train', 'val', 'test']:
        split_path = os.path.join(base_source_dir, split)
        if not os.path.exists(split_path):
            print(f"Priečinok {split} nebol nájdený, preskakujem...")
            continue

        for cls in os.listdir(split_path):
            source_folder = os.path.join(split_path, cls)
            if not os.path.isdir(source_folder):
                continue

            dest_hsv = os.path.join(target_root_hsv, split, cls)
            dest_inpaint = os.path.join(target_root_inpaint, split, cls)

            os.makedirs(dest_hsv, exist_ok=True)
            os.makedirs(dest_inpaint, exist_ok=True)

            if cls.lower() == "anomaly":
                img_paths = []
                for bad_id in range(1, 7):
                    bad_folder = os.path.join(source_folder, f"bad{bad_id}")
                    img_paths.extend(glob.glob(os.path.join(bad_folder, "*.*")))
            else:
                img_paths = glob.glob(os.path.join(source_folder, "*.*"))

            print(f"Spracovávam {split}/{cls} ({len(img_paths)} obrázkov)...")

            for img_path in img_paths:
                try:
                    img_np = np.array(Image.open(img_path).convert("RGB"))

                    processed_hsv = remove_glare_hsv(img_np)
                    processed_inpaint, _ = remove_glare_inpaint(img_np)

                    filename = os.path.basename(img_path)
                    name, _ = os.path.splitext(filename)

                    if cls.lower() == "anomaly":
                        bad_folder_name = os.path.basename(os.path.dirname(img_path))
                        name = f"{bad_folder_name}_{name}"

                    Image.fromarray(processed_hsv).save(os.path.join(dest_hsv, f"{name}.png"))
                    Image.fromarray(processed_inpaint).save(os.path.join(dest_inpaint, f"{name}.png"))

                except Exception as e:
                    print(f"Chyba pri {img_path}: {e}")

    print("\n--- HOTOVO! DATASETY SÚ ULOŽENÉ ---")
    print(f"-> {target_root_hsv}")
    print(f"-> {target_root_inpaint}")


# 3. VIZUALIZÁCIA (2x GOOD, 2x DEFEKT)
def show_visual_comparison(base_source_dir):
    print("\nGenerujem vizuálne porovnanie: 2x GOOD a 2x DEFEKT...")

    good_paths = (
        glob.glob(os.path.join(base_source_dir, "val", "good", "*.*")) +
        glob.glob(os.path.join(base_source_dir, "test", "good", "*.*")) +
        glob.glob(os.path.join(base_source_dir, "train", "good", "*.*"))
    )

    defect_paths = []

    for bad_id in range(1, 7):
        defect_paths.extend(
            glob.glob(os.path.join(base_source_dir, "val", "anomaly", f"bad{bad_id}", "*.*"))
        )
        defect_paths.extend(
            glob.glob(os.path.join(base_source_dir, "test", "anomaly", f"bad{bad_id}", "*.*"))
        )

    # Menej obrázkov, aby sa výstup zmestil do dokumentácie
    num_good = min(2, len(good_paths))
    num_defect = min(2, len(defect_paths))

    sampled_good = random.sample(good_paths, num_good) if num_good > 0 else []
    sampled_defect = random.sample(defect_paths, num_defect) if num_defect > 0 else []

    all_sampled_paths = sampled_good + sampled_defect
    num_samples = len(all_sampled_paths)

    if num_samples == 0:
        print("Chyba: Nenašli sa žiadne obrázky. Skontroluj, či sú cesty k zložkám správne.")
        return

    # Menšia výška výsledného obrázka
    fig, axes = plt.subplots(
        num_samples,
        4,
        figsize=(18, 4 * num_samples),
        squeeze=False
    )

    for i, img_path in enumerate(all_sampled_paths):
        img_pil = Image.open(img_path).convert("RGB")
        img_np = np.array(img_pil)

        res_inpaint, generated_mask = remove_glare_inpaint(img_np)
        res_hsv = remove_glare_hsv(img_np)

        img_name = os.path.basename(img_path)
        folder_name = os.path.basename(os.path.dirname(img_path))

        if "good" in folder_name.lower():
            status = "OK (Good)"
            color = "green"
        else:
            status = f"DEFEKT ({folder_name})"
            color = "red"

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(
            f"Originál [{status}]\n({img_name})",
            color=color,
            fontweight='bold',
            fontsize=10
        )
        axes[i, 0].axis('off')

        axes[i, 1].imshow(generated_mask, cmap='gray')
        axes[i, 1].set_title("Maska Inpaint", fontsize=10)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(res_inpaint)
        axes[i, 2].set_title("Inpainting", fontsize=10)
        axes[i, 2].axis('off')

        axes[i, 3].imshow(res_hsv)
        axes[i, 3].set_title("HSV úprava", fontsize=10)
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.show()

# 4. SPUSTENIE
BASE_SOURCE_DIR = r"C:\ae_fcdd"

#generate_datasets(
#    base_source_dir=BASE_SOURCE_DIR,
#    target_root_hsv=r"C:\dataset_basic_hsv",
#    target_root_inpaint=r"C:\dataset_basic_inpaint"
#)

show_visual_comparison(BASE_SOURCE_DIR)

In [5]:
# Výber predspracovaného datasetu a načítanie train/val/test dát.
import os
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

METHOD_TO_TRAIN = "basic_inpaint"  

if METHOD_TO_TRAIN == "basic_hsv":
    root_dir = r"C:\dataset_basic_hsv"
elif METHOD_TO_TRAIN == "basic_inpaint":
    root_dir = r"C:\dataset_basic_inpaint"

transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor()
])

transform_eval = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor()
])

dataset_train_full = datasets.ImageFolder(
    os.path.join(root_dir, "train"),
    transform=transform_train
)

good_idx_train = dataset_train_full.class_to_idx["good"]

train_good_indices = [
    i for i, label in enumerate(dataset_train_full.targets)
    if label == good_idx_train
]

dataset_train = Subset(dataset_train_full, train_good_indices)

train_loader = DataLoader(
    dataset_train,
    batch_size=batch_size,
    shuffle=True
)

dataset_val = datasets.ImageFolder(
    os.path.join(root_dir, "val"),
    transform=transform_eval
)

good_idx_val = dataset_val.class_to_idx["good"]

val_good_indices = [
    i for i, label in enumerate(dataset_val.targets)
    if label == good_idx_val
]

val_def_indices = [
    i for i, label in enumerate(dataset_val.targets)
    if label != good_idx_val
]

val_good_loader = DataLoader(
    Subset(dataset_val, val_good_indices),
    batch_size=batch_size,
    shuffle=False
)

val_def_loader = DataLoader(
    Subset(dataset_val, val_def_indices),
    batch_size=batch_size,
    shuffle=False
)

dataset_test = datasets.ImageFolder(
    os.path.join(root_dir, "test"),
    transform=transform_eval
)

good_idx_test = dataset_test.class_to_idx["good"]

test_good_indices = [
    i for i, label in enumerate(dataset_test.targets)
    if label == good_idx_test
]

test_def_indices = [
    i for i, label in enumerate(dataset_test.targets)
    if label != good_idx_test
]

test_good_loader = DataLoader(
    Subset(dataset_test, test_good_indices),
    batch_size=batch_size,
    shuffle=False
)

test_def_loader = DataLoader(
    Subset(dataset_test, test_def_indices),
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# Tréning autoencodera na zvolenom predspracovanom datasete a uloženie najlepšieho modelu.
import copy
import torch
import torch.nn as nn
import torch.optim as optim

save_model_name = f"best_autoencoder_{METHOD_TO_TRAIN}.pth"
print(f"Najlepší model sa bude ukladať ako: {save_model_name}")

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

history_train_loss = []
history_val_loss = []
best_val_loss = float("inf")
best_model_wts = copy.deepcopy(model.state_dict())
early_stop_counter = 0

for epoch in range(epochs):
    # --- TRÉNOVANIE ---
    model.train()
    train_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        outputs = model(images)
        loss = loss_fn(outputs, images)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
    train_loss /= len(train_loader.dataset)
    
    # --- VALIDÁCIA ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, _ in val_good_loader:
            images = images.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, images)
            val_loss += loss.item() * images.size(0)
    val_loss /= len(val_good_loader.dataset)
    
    history_train_loss.append(train_loss)
    history_val_loss.append(val_loss)
    scheduler.step(val_loss)
    
    print(f"Epoch [{epoch+1}/{epochs}] - train_loss: {train_loss:.4f} - val_good_loss: {val_loss:.4f}")
    
    # --- UKLADANIE NAJLEPŠIEHO MODELU ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), rf"C:\BestModel\{save_model_name}")  
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        
    if early_stop_counter >= patience:
        print(f"Early stopping na epoche {epoch+1}")
        break

# Načítanie najlepších váh späť do modelu po skončení tréningu
model.load_state_dict(torch.load(rf"C:\BestModel\{save_model_name}"))
print("Hotovo! Najlepší model je načítaný a pripravený na testovanie.")

In [ ]:
# Vykreslenie priebehu trénovacej a validačnej chyby.
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(history_train_loss) + 1), history_train_loss, label='Trénovacia Loss', color='blue')
plt.plot(range(1, len(history_val_loss) + 1), history_val_loss, label='Validačná Loss (Good)', color='green')
plt.title('Priebeh trénovania (Výlučne MSE)')
plt.xlabel('Epocha')
plt.ylabel('Hodnota Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Výpočet MSE anomaly score.
# Pre každý obrázok sa vypočíta priemerná rekonštrukčná chyba MSE.
def get_anomaly_scores(loader, model, device):
    scores = []
    model.eval()

    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            reconstructions = model(images)

            error_map = torch.square(reconstructions - images)
            anomaly_score = error_map.mean(dim=(1, 2, 3))

            scores.extend(anomaly_score.cpu().numpy())

    return np.array(scores)


# Výpočet validačných MSE skóre.
# Anomaly score je MSE, F1-score sa používa iba na výber najlepšieho threshold-u.
val_normal_scores = get_anomaly_scores(val_good_loader, model, device)
val_defect_scores = get_anomaly_scores(val_def_loader, model, device)

y_val_true = np.concatenate([
    np.zeros(len(val_normal_scores), dtype=int),
    np.ones(len(val_defect_scores), dtype=int)
])

y_val_scores = np.concatenate([
    val_normal_scores,
    val_defect_scores
])

thresholds = np.linspace(y_val_scores.min(), y_val_scores.max(), 500)

best_thresh = None
best_f1 = -1

for thr in thresholds:
    y_val_pred = (y_val_scores > thr).astype(int)
    f1 = f1_score(y_val_true, y_val_pred, zero_division=0)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thr

RECON_THRESH = best_thresh

print("=" * 60)
print("THRESHOLD Z VALIDÁCIE")
print("=" * 60)
print("Anomaly score: MSE")
print(f"Best threshold: {RECON_THRESH:.6f}")
print(f"Best validation F1: {best_f1:.6f}")

plt.figure(figsize=(10, 5))
plt.hist(val_normal_scores, bins=40, alpha=0.6, color='green', label='VAL good', edgecolor='black')
plt.hist(val_defect_scores, bins=40, alpha=0.6, color='red', label='VAL defect', edgecolor='black')
plt.axvline(RECON_THRESH, color='blue', linestyle='--', linewidth=2, label=f'MSE threshold = {RECON_THRESH:.4f}')
plt.title('Distribúcia anomaly score (MSE)')
plt.xlabel('Anomaly score (MSE)')
plt.ylabel('Počet obrázkov')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Zobrazenie dobrých vzoriek s najvyššou rekonštrukčnou chybou.
def show_worst_good_images(model, loader, device, num_images=5, num_cols=5):
    print("=" * 60)
    print("ANALÝZA NAJHORŠÍCH 'DOBRÝCH' OBRAZOV (Odlesky/Problémy)")
    print("=" * 60)
    model.eval()
    
    image_data = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            reconstructions = model(images)
            error_map = torch.square(reconstructions - images)
            anomaly_score = error_map.mean(dim=(1, 2, 3))
            
            for i in range(images.size(0)):
                image_data.append({
                    'orig': images[i].cpu(),
                    'recon': reconstructions[i].cpu(),
                    'error_map': error_map[i].cpu(),
                    'score': anomaly_score[i].item()
                })
                
    image_data.sort(key=lambda x: x['score'], reverse=True)
    top_worst = image_data[:num_images]

    fig, axes = plt.subplots(3, num_cols, figsize=(4 * num_cols, 12))
    fig.suptitle("Najhoršie rekonštruované 'DOBRÉ' obrazy (odlesky)", fontsize=16)
    
    for i, data in enumerate(top_worst):
        col = i
        
        img_np = data['orig'].squeeze().numpy()
        rec_np = data['recon'].squeeze().numpy()
        err_np = data['error_map'].squeeze().numpy()
        
        axes[0, col].imshow(img_np, cmap='gray')
        axes[0, col].set_title(f"Orig\nMSE: {data['score']:.4f}", fontsize=10)
        axes[0, col].axis('off')
        
        axes[1, col].imshow(rec_np, cmap='gray')
        axes[1, col].set_title("Rekonštrukcia", fontsize=10)
        axes[1, col].axis('off')
        
        axes[2, col].imshow(img_np, cmap='gray')
        normalized_err_np = err_np / np.max(err_np) if np.max(err_np) > 0 else err_np
        axes[2, col].imshow(normalized_err_np, cmap='jet', alpha=0.5) 
        axes[2, col].set_title("Heatmapa MSE", fontsize=10)
        axes[2, col].axis('off')
        
    # Ak by bolo obrázkov menej ako 5, zvyšné sloty sa skryjú
    for col in range(len(top_worst), num_cols):
        axes[0, col].axis('off')
        axes[1, col].axis('off')
        axes[2, col].axis('off')
        
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

show_worst_good_images(model, val_good_loader, device, num_images=5)

In [ ]:
# Vyhodnotenie testovacej množiny pre viacero hodnôt prahu.
test_normal_scores = get_anomaly_scores(test_good_loader, model, device)
test_defect_scores = get_anomaly_scores(test_def_loader, model, device)

y_test_true = np.concatenate([
    np.zeros(len(test_normal_scores), dtype=int),
    np.ones(len(test_defect_scores), dtype=int)
])
y_test_scores = np.concatenate([test_normal_scores, test_defect_scores])

thresh_1 = RECON_THRESH
thresh_2 = RECON_THRESH * 2 
thresh_3 = RECON_THRESH * 3

print("=" * 60)
print("POROVNANIE VÝSLEDKOV PRE RÔZNE HODNOTY THRESHOLD (Test Set)")
print("=" * 60)
table_header = f"| {'Threshold':^12} | {'Precision':^10} | {'Recall (Defekty)':^18} | {'F1-Score':^10} | {'Falošné Detekcie (FP)':^23} |"
print(table_header)
print("|" + "-"*14 + "|" + "-"*12 + "|" + "-"*20 + "|" + "-"*12 + "|" + "-"*25 + "|")

for t in [thresh_1, thresh_2, thresh_3]:
    y_pred_t = (y_test_scores > t).astype(int)
    cm_t = confusion_matrix(y_test_true, y_pred_t)
    fp_count = cm_t[0, 1] 
    
    prec = precision_score(y_test_true, y_pred_t, zero_division=0)
    rec = recall_score(y_test_true, y_pred_t, zero_division=0)
    f1_s = f1_score(y_test_true, y_pred_t, zero_division=0)
    
    row = f"| {t:^12.5f} | {prec:^10.4f} | {rec:^18.4f} | {f1_s:^10.4f} | {fp_count:^23} |"
    print(row)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
thresholds_list = [thresh_1, thresh_2, thresh_3]
titles = [f'Threshold 1: {thresh_1:.4f}', f'Threshold 2: {thresh_2:.4f}', f'Threshold 3: {thresh_3:.4f}']

for i, t in enumerate(thresholds_list):
    y_pred_t = (y_test_scores > t).astype(int)
    cm_t = confusion_matrix(y_test_true, y_pred_t)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_t, display_labels=['OK', 'DEFEKT'])
    disp.plot(cmap='Blues', ax=axes[i], values_format='d', colorbar=False)
    axes[i].set_title(titles[i])

plt.tight_layout()
plt.show()

In [ ]:
# Vizualizácia defektných vzoriek: originál, rekonštrukcia a MSE heatmapa.
def show_anomaly_heatmaps(model, dataset, subset_loader, device, num_images=5, num_cols=5):
    model.eval()
    subset_indices = subset_loader.dataset.indices
    num_images = min(num_images, len(subset_indices), num_cols)

    random_indices = random.sample(subset_indices, num_images)
    random_images = []
    random_labels = []
    
    for idx in random_indices:
        img, label = dataset[idx]
        random_images.append(img)
        random_labels.append(dataset.classes[label])
        
    images = torch.stack(random_images).to(device)

    with torch.no_grad():
        reconstructions = model(images)
        error_maps = torch.square(reconstructions - images)

    fig, axes = plt.subplots(
        3,
        num_cols,
        figsize=(4 * num_cols, 12)
    )
    
    for i in range(num_images):
        col = i

        img_np = images[i].cpu().squeeze().numpy()
        rec_np = reconstructions[i].cpu().squeeze().numpy()
        err_np = error_maps[i].cpu().squeeze().numpy()
        
        axes[0, col].imshow(img_np, cmap='gray')
        axes[0, col].set_title(f"{random_labels[i]} | originál")
        axes[0, col].axis('off')
        
        axes[1, col].imshow(rec_np, cmap='gray')
        axes[1, col].set_title("Rekonštrukcia")
        axes[1, col].axis('off')
        
        axes[2, col].imshow(img_np, cmap='gray')
        axes[2, col].imshow(err_np, cmap='jet', alpha=0.45)
        axes[2, col].set_title("Heatmapa MSE")
        axes[2, col].axis('off')
        
    # Ak by bolo obrázkov menej ako 5, zvyšné sloty sa skryjú
    for col in range(num_images, num_cols):
        axes[0, col].axis('off')
        axes[1, col].axis('off')
        axes[2, col].axis('off')
        
    plt.tight_layout()
    plt.show()


show_anomaly_heatmaps(model, dataset_test, test_def_loader, device, num_images=5)